# RQ3: training data — training

How the composition of training episodes changes the development of transfer: a full pass over every table (E7), small versus count-matched large tables (E8) and the episode size (E9), the latter also at equal numbers of processed rows.

Monitor points use one fixed context/query split per table (up to 10,000 rows, two preprocessing views); they show how the effect develops and are not the five-fold evaluation.

In [1]:
%matplotlib inline
import sys
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import pandas as pd
from src.visualize import style, campaign as cp, campaign2 as c2, results2 as r2
from src.visualize.figures import FigureSaver
style.apply()
sink = FigureSaver('rq3/01_training')
report = cp.NotebookReport('RQ3: training data — training', sink=sink)
frame = c2.load_trajectories(['reference', 'exposure', 'table_size', 'episode_size'])
curves = c2.monitor_curves(c2.monitor_changes(frame))

## 1. Design

The arms of this research question: what each changes versus the reference recipe, the tasks and the number of training runs.

In [2]:
report.table('Arms of RQ3', c2.design_table(3))
report.add('1. Design', 'Each arm changes only the listed settings; E1 is the shared control of E7. E8 compares its two corpora with each other; E9 trains on the tables above 10,000 rows.')

,experiment,slug,arm,track,change,models,runs
0,E7,exposure,0,pd,exposure=full_pass,4,16
1,E7,exposure,0,lgd,exposure=full_pass,4,16
2,E8,table_size,0,pd,corpus=small_only,4,16
3,E8,table_size,1,pd,corpus=large_matched,4,16
4,E9,episode_size,0,pd,"corpus=tables_above_10000_rows, episode_rows=2048",4,16
5,E9,episode_size,1,pd,"corpus=tables_above_10000_rows, episode_rows=4096",4,16
6,E9,episode_size,2,pd,"corpus=tables_above_10000_rows, episode_rows=8192",4,16
7,E9,episode_size,3,pd,"corpus=tables_above_10000_rows, episode_rows=1...",2,8
8,E9,episode_size,4,pd,corpus=tables_above_10000_rows,4,16


## 2. Exposure and table size (E7, E8)

Question: does the held-out credit score develop differently when every row is visited, or when only small (or the same number of large) tables are used?

In [3]:
for track in ('pd', 'lgd'):
    slugs = ['reference', 'exposure', 'table_size']
    settings = pd.concat([c2.arm_settings(slug, track) for slug in slugs])
    arms = r2.arm_styles(settings, ordered=False)
    sub = c2.select(curves, track=track, slug=slugs, scope='test')
    fig = r2.arm_trajectories([dict(curves=sub, x='successful_updates', ylabel='Held-out credit: Δ ' + r2.SCORE[track])],
                              arms, title='Training-data arms during training (E7, E8)')
    cp.show(sink, c2.pages(fig, 'data_monitor',
        'Change from update 0 of the equal-table mean held-out credit monitor score, averaged over the four partitions, '
        'per base model (columns), for the reference (E1, black) and the training-data arms: '
        'a full pass over every table with one update per chunk (E7), and only tables up to 10,000 rows versus the '
        'same number of tables above 10,000 rows (E8, PD). Monitor: one fixed context/query split of up to 10,000 rows per '
        'table, not the five-fold evaluation.'), track=track)
    report.table(track.upper() + ' change at the last monitor point (E1, E7-E8)', c2.curve_endpoints(sub))
report.add('2. Exposure and table size (E7, E8)', c2.evidence_note('E7-E8 monitor curves', c2.select(curves, slug=['exposure', 'table_size'])))

No measurements available for this section yet.


""


No measurements available for this section yet.


""


## 3. Episode size at equal updates and at equal rows (E9)

Question: do larger training episodes help, compared at the same number of updates (top) and at the same number of processed rows (bottom; milestones at 1, 2, 5, 10, 20 and 50 million rows)?

In [4]:
sub = c2.select(curves, track='pd', slug='episode_size', scope='test')
by_rows = c2.select(c2.monitor_curves(c2.monitor_changes(c2.select(frame, slug='episode_size'), x='processed_rows', scopes=('test',)), x='processed_rows'), track='pd')
arms = r2.arm_styles(c2.arm_settings('episode_size', 'pd'), color_by='episode_rows')
fig = r2.arm_trajectories([dict(curves=sub, x='successful_updates', ylabel='Δ AUC by updates'),
                           dict(curves=by_rows, x='processed_rows', ylabel='Δ AUC by rows')],
                          arms, title='Training-episode size at equal updates and equal rows (E9)')
cp.show(sink, c2.pages(fig, 'episode_size_monitor',
    'Change from update 0 of the equal-table mean held-out credit monitor AUC for training episodes of 2,048 to 16,384 '
    'rows (colour; 16,384 for TabPFN v3 and TabICLv2 only), all on the corpus of tables above 10,000 rows, per base model '
    '(columns): against successful updates (top) and against rows processed at the 1, 2, 5, 10, 20 and 50 million-row '
    'milestones (bottom, log scale). Averaged over the four partitions; one fixed context/query split of up to 10,000 '
    'rows per table, not the five-fold evaluation.'), track='pd')
report.table('E9 change at the last monitor point', c2.curve_endpoints(sub))
report.table('E9 change at each rows-processed milestone', by_rows)
report.add('3. Episode size at equal updates and at equal rows (E9)', c2.evidence_note('E9 monitor curves', sub))

No measurements available for this section yet.


""


""


## 4. Training cost

Wall-clock hours, rows processed and peak reserved GPU memory per arm and model; E7 and E9 change the data seen per update and therefore the cost.

In [5]:
report.table('Training cost per arm and model (E1, E7-E9)', c2.resource_summary(c2.run_resources(frame)))
report.add('4. Training cost', c2.evidence_note('training cost records', frame))

""


## Summary

Every displayed table and figure, in section order.

In [6]:
print(report.summary(sink))

RQ3: training data — training

1. Design
Each arm changes only the listed settings; E1 is the shared control of E7. E8 compares its two corpora with each other; E9 trains on the tables above 10,000 rows.

Table: Arms of RQ3
CSV (first column: row index)
row,experiment,slug,arm,track,change,models,runs
0,E7,exposure,0,pd,exposure=full_pass,4,16
1,E7,exposure,0,lgd,exposure=full_pass,4,16
2,E8,table_size,0,pd,corpus=small_only,4,16
3,E8,table_size,1,pd,corpus=large_matched,4,16
4,E9,episode_size,0,pd,"corpus=tables_above_10000_rows, episode_rows=2048",4,16
5,E9,episode_size,1,pd,"corpus=tables_above_10000_rows, episode_rows=4096",4,16
6,E9,episode_size,2,pd,"corpus=tables_above_10000_rows, episode_rows=8192",4,16
7,E9,episode_size,3,pd,"corpus=tables_above_10000_rows, episode_rows=16384",2,8
8,E9,episode_size,4,pd,corpus=tables_above_10000_rows,4,16

2. Exposure and table size (E7, E8)
No results yet for E7-E8 monitor curves; the figure is drawn once they exist.

No measurements availabl